In [ ]:
import cv2
import torch
from torchvision import models,transforms
from PIL import Image
from collections import Counter

STATES = ['Chatter', 'Normal', 'RoughCut', 'ToolBroken']

def generate_marked_video(video_path, output_path, model_path):
    # 1. 載入模型
    model = models.resnet18()
    model.fc = torch.nn.Linear(model.fc.in_features, 4)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # 2. 讀取影片資訊
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # 設定影片輸出路徑與編碼
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # 3. 穩定邏輯變數
    state_buffer = []
    buffer_size = 10 # 緩衝幀數，避免文字閃爍太快
    
    print(f"開始生成標註影片：{output_path}...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        # 影像預處理
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        input_tensor = preprocess(Image.fromarray(img_rgb)).unsqueeze(0)

        # 4. 進行時時刻刻的預測
        with torch.no_grad():
            output = model(input_tensor)
            pred_idx = torch.argmax(output).item()
            raw_state = STATES[pred_idx]

        # 穩定化文字顯示
        state_buffer.append(raw_state)
        if len(state_buffer) > buffer_size:
            state_buffer.pop(0)
        display_state = Counter(state_buffer).most_common(1)[0][0]

        # 5. 在畫面上標註文字 (Process State)
        cv2.rectangle(frame, (40, 20), (450, 90), (0, 0, 0), -1) 
        cv2.putText(frame, f"STATE: {display_state}", (50, 70), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

        # 6. 寫入每一影格到輸出影片
        out.write(frame)

    cap.release()
    out.release()
    print("標註影片輸出完成！")

# 執行標註任務
generate_marked_video(r"D:\yang_neural_network\mid_term\TestMachining.mp4", 'marked_result_10resnet18.mp4', 'normal_ephos10_machining_model.pth')

開始生成標註影片：marked_result.mp4...
標註影片輸出完成！


## 問題
根據TestMachining.mp4，將其標上狀態，如:Chatter , rough cut , Tool broken,etc.

## 方法:
step1:使用期中考Q1Q2訊練好的模型，ResNet18 作為特徵提取器。

step2: while cap.isOpened()對每一幀做捕捉。

step3:避免狀態跳動頻繁，設buffer為10，紀錄最近出現最多次數作為顯示。